# Fine-tuning Ember-NNUE with nnue-pytorch

This notebook fine-tunes the **external network for the Ember engine** in the binary container
format V2 (magic `0x6A448AFA`):

| Parameter           | Value                                     |
|---------------------|-------------------------------------------|
| `Full_Threats`      | 60,720 inputs, i8 weights                 |
| `HalfKAv2_hm^` (PSQ)| 22,528 inputs, i16 weights                |
| `L1` (hidden)       | 1024 (SCReLU)                             |
| `Affine` stacks     | 8 buckets × (32 → 32 → 1), SCCReLU        |
| PSQT buckets        | 8                                         |
| ARCH_HASH           | `0x0256acdf`                              |
| FT_HEADER_HASH      | `0x6165ddc9`                              |
| STACK_HASH          | `0x63337116`                              |

Dataset: [farseerT74](https://huggingface.co/datasets/official-stockfish/master-binpacks/blob/main/farseerT74.binpack)
(~20 GB). The notebook downloads it, installs dependencies,
and builds the C++ data loader.

**Run order**
1. Runtime → Change runtime type → **GPU** (T4 / A100 / L4).
2. Run all cells in order. Downloading the data takes a while.
3. One superbatch = 100,000,000 positions = 6104 batches × 16384. Set the number
   of additional superbatches; auto-save runs every **3** by default.
4. If the session dies, set `RESUME_CHECKPOINT` to this run's exact checkpoint.
   After confirming the trainer is gone, set `RECOVER_DEAD_LAUNCH=True`.
   Keep the base checkpoint and run outputs separate on Google Drive.


## 0. Environment and GPU


In [ ]:
!nvidia-smi
import torch, sys
print("Python", sys.version)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. nnue-pytorch (pinned commit)

Uses commit `a7830b2a91d15f6d3214bd21b1a6cc5cf7701b82` — the last one before
`PP_3Wide` was added. It has `Full_Threats` = 60,720 inputs and the default
`Full_Threats+HalfKAv2_hm^`, i.e. exactly the architecture Ember expects.


In [ ]:
%%bash
set -e
cd /content
if [ ! -d /content/nnue-pytorch/.git ]; then
  echo "Cloning nnue-pytorch..."
  git clone https://github.com/official-stockfish/nnue-pytorch /content/nnue-pytorch
fi
cd /content/nnue-pytorch
git fetch --quiet --all 2>/dev/null || true
git checkout -q a7830b2a91d15f6d3214bd21b1a6cc5cf7701b82
git rev-parse HEAD
echo "OK"


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq >/dev/null 2>&1
apt-get install -y -qq cmake g++ >/dev/null 2>&1
pip install -q -r /content/nnue-pytorch/requirements.txt
pip uninstall -y -q tensorflow tensorflow-cpu tf-nightly tf-keras 2>/dev/null || true
python -c "import torch, tyro, lightning; print('torch', torch.__version__, '| tyro OK | lightning', lightning.__version__)"


### Building the C++ data loader

Without `libtraining_data_loader.so` training will not start — it is linked via
ctypes from `./build/`, so training must be launched from the `/content/nnue-pytorch` directory.


In [ ]:
%%bash
set -e
cd /content/nnue-pytorch
cmake -S data_loader/cpp -B build -DCMAKE_BUILD_TYPE=Release > /tmp/cmake_cfg.log 2>&1
cmake --build build -j"$(nproc)" > /tmp/cmake_build.log 2>&1
ls -la build/libtraining_data_loader* build/training_data_loader* 2>/dev/null | head
cd /content/nnue-pytorch && python -c "import data_loader; print('data_loader OK')"


## 2. Dataset `farseerT74.binpack` (~20 GB)

The file is downloaded to `/content/data` (runtime disk). An incomplete download
is kept separately and resumed; exact size and SHA-256 are verified before use.

In [ ]:
%%bash
set -euo pipefail
URL="https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/farseerT74.binpack?download=true"
DATASET="/content/data/farseerT74.binpack"
PART="${DATASET}.part"
EXPECTED_SIZE=20144023865
EXPECTED_SHA256="cebf6e5aa62a0df447f3748c90a542ded13105c873a1a819d7f71bdf041ca8cb"

mkdir -p /content/data
if [ -f "$DATASET" ]; then
  test "$(stat -c %s "$DATASET")" -eq "$EXPECTED_SIZE"
  printf '%s  %s\n' "$EXPECTED_SHA256" "$DATASET" | sha256sum --check -
  echo "Verified dataset is already present."
  exit 0
fi
PART_SIZE=0
if [ -f "$PART" ]; then
  PART_SIZE="$(stat -c %s "$PART")"
  if [ "$PART_SIZE" -gt "$EXPECTED_SIZE" ]; then
    echo "Partial file exceeds expected size: $PART" >&2
    exit 1
  fi
fi
if [ "$PART_SIZE" -lt "$EXPECTED_SIZE" ]; then
  echo "Downloading or resuming the file..."
  curl --fail --location --continue-at - --retry 5 --retry-all-errors --retry-delay 5 --output "$PART" "$URL"
fi
test "$(stat -c %s "$PART")" -eq "$EXPECTED_SIZE"
printf '%s  %s\n' "$EXPECTED_SHA256" "$PART" | sha256sum --check -
mv "$PART" "$DATASET"
echo "Download and verification complete!"
ls -lh "$DATASET"

## 3. Training

**A superbatch is one epoch.** Parameters are set in the cell below. Specify the exact
base checkpoint and its SHA-256. To continue an interrupted run, explicitly set its
checkpoint in `RESUME_CHECKPOINT`; the notebook never selects by modification time.

`ADDITIONAL_EPOCHS` is the number of new superbatches after the base checkpoint epoch.
The absolute `--max-epochs` target is frozen in the run manifest. Smoke mode always writes
to a separate directory with a `-smoke` suffix. Obtain the SHA-256 with
`sha256sum /exact/path/last.ckpt`.


In [ ]:
import os

os.environ['NNUE_REPO']   = '/content/nnue-pytorch'
os.environ['DATASET']     = '/content/data/farseerT74.binpack'
os.environ['DATASET_SHA256'] = 'cebf6e5aa62a0df447f3748c90a542ded13105c873a1a819d7f71bdf041ca8cb'
os.environ['DRIVE_DIR']   = '/content/drive/MyDrive/Ember_Networks'
os.environ['RUN_NAME']    = 'ember_v2_farseer_finetune_2026'
os.environ['BASE_CHECKPOINT'] = '/content/drive/MyDrive/Ember_Networks/REPLACE_ME/last.ckpt'
os.environ['BASE_CHECKPOINT_SHA256'] = 'REPLACE_WITH_64_HEX_SHA256'
os.environ['RESUME_CHECKPOINT'] = ''
os.environ['RECOVER_DEAD_LAUNCH'] = 'False'
os.environ['ADDITIONAL_EPOCHS'] = '300'
os.environ['SAVE_EVERY']  = '3'
os.environ['BATCH_SIZE']  = '16384'
os.environ['EPOCH_SIZE']  = '100000000'
os.environ['NUM_WORKERS'] = '4'
os.environ['SMOKE']       = 'False'

print("config:", {k: os.environ[k] for k in ['NNUE_REPO','DATASET','DATASET_SHA256','DRIVE_DIR','RUN_NAME','BASE_CHECKPOINT','BASE_CHECKPOINT_SHA256','RESUME_CHECKPOINT','RECOVER_DEAD_LAUNCH','ADDITIONAL_EPOCHS','SAVE_EVERY','BATCH_SIZE','EPOCH_SIZE','NUM_WORKERS','SMOKE']})


In [ ]:
import datetime as dt
import hashlib
import json
import os
import signal
import socket
import subprocess
import sys
import tempfile
import uuid
from pathlib import Path

os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
os.environ["CUDNN_DETERMINISTIC"] = "0"
os.environ["CUDNN_BENCHMARK"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PL_TORCH_BACKEND"] = "torch"
os.environ["LIGHTNING_PRECISION"] = "16-mixed"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require_sha256(path, expected, label):
    expected = expected.strip().lower()
    if len(expected) != 64 or any(char not in '0123456789abcdef' for char in expected):
        raise ValueError(f'{label} SHA-256 must be exactly 64 hexadecimal characters')
    actual = sha256_file(path)
    if actual != expected:
        raise ValueError(f'{label} SHA-256 mismatch: expected {expected}, got {actual}')
    return actual

def checkpoint_epoch(path):
    import torch
    try:
        checkpoint = torch.load(str(path), map_location='cpu', weights_only=False)
    except TypeError:
        checkpoint = torch.load(str(path), map_location='cpu')
    epoch = checkpoint.get('epoch') if isinstance(checkpoint, dict) else None
    if isinstance(epoch, bool) or not isinstance(epoch, int) or epoch < 0:
        raise ValueError(f'checkpoint has no valid nonnegative epoch: {path}')
    return epoch

def derive_target_max_epochs(base_epoch, additional_epochs):
    if base_epoch < 0:
        raise ValueError('base checkpoint epoch must be nonnegative')
    if additional_epochs <= 0:
        raise ValueError('ADDITIONAL_EPOCHS must be positive')
    return base_epoch + 1 + additional_epochs

def validate_resume_epoch(base_epoch, resume_epoch, target_max_epochs):
    if resume_epoch < base_epoch:
        raise ValueError('resume checkpoint predates the declared base checkpoint')
    if resume_epoch + 1 >= target_max_epochs:
        raise ValueError('resume checkpoint has already reached the frozen epoch target')

def atomic_write_json(path, value):
    descriptor, temporary = tempfile.mkstemp(
        prefix=f'.{path.name}.', suffix='.tmp', dir=path.parent
    )
    try:
        with os.fdopen(descriptor, 'w', encoding='utf-8') as stream:
            json.dump(value, stream, indent=2, sort_keys=True)
            stream.write('\n')
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

def process_identity(pid):
    process_dir = Path('/proc') / str(pid)
    try:
        stat_tail = (process_dir / 'stat').read_text().rsplit(')', 1)[1].split()
        command = (process_dir / 'cmdline').read_bytes().split(b'\0')
    except FileNotFoundError:
        return None
    except (OSError, IndexError) as error:
        raise RuntimeError(f'cannot identify active trainer pid {pid}') from error
    return {
        'pid': pid,
        'start_ticks': stat_tail[19],
        'command': [part.decode('utf-8', errors='replace') for part in command if part],
    }

def archive_launch(output_root, value):
    history_dir = output_root / 'launch-history'
    history_dir.mkdir(exist_ok=True)
    launch_id = value.get('launch_id')
    if not launch_id:
        encoded = json.dumps(value, sort_keys=True).encode('utf-8')
        launch_id = hashlib.sha256(encoded).hexdigest()
    atomic_write_json(history_dir / f'{launch_id}.json', value)

NNUE_REPO = Path(os.environ['NNUE_REPO']).expanduser().resolve()
DATASET = Path(os.environ['DATASET']).expanduser().resolve()
DRIVE_DIR = Path(os.environ['DRIVE_DIR']).expanduser().resolve()
RUN_NAME = os.environ['RUN_NAME'].strip()
BASE_CHECKPOINT = Path(os.environ['BASE_CHECKPOINT']).expanduser().resolve()
BASE_CHECKPOINT_SHA256 = os.environ['BASE_CHECKPOINT_SHA256']
DATASET_SHA256 = os.environ['DATASET_SHA256']
RESUME_CHECKPOINT = os.environ.get('RESUME_CHECKPOINT', '').strip()
recover_dead_launch = os.environ.get('RECOVER_DEAD_LAUNCH', 'False').strip().lower() in ('1','true','yes')
EXPECTED_NNUE_REVISION = 'a7830b2a91d15f6d3214bd21b1a6cc5cf7701b82'

additional_epochs = int(os.environ['ADDITIONAL_EPOCHS'])
save_every   = int(os.environ['SAVE_EVERY'])
batch_size   = int(os.environ['BATCH_SIZE'])
epoch_size   = int(os.environ['EPOCH_SIZE'])
num_workers  = int(os.environ['NUM_WORKERS'])
smoke        = os.environ.get('SMOKE', 'False').strip().lower() in ('1','true','yes')

if not RUN_NAME or '/' in RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME must be one non-empty path component')
if additional_epochs <= 0:
    raise ValueError('ADDITIONAL_EPOCHS must be positive')
nnue_revision = subprocess.check_output(
    ['git', '-C', str(NNUE_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
if nnue_revision != EXPECTED_NNUE_REVISION:
    raise ValueError(f'nnue-pytorch revision mismatch: {nnue_revision}')
tracked_changes = subprocess.check_output(
    ['git', '-C', str(NNUE_REPO), 'status', '--porcelain', '--untracked-files=no'],
    text=True,
).strip()
if tracked_changes:
    raise ValueError('nnue-pytorch has tracked local changes')
if not DATASET.is_file():
    raise FileNotFoundError('dataset not found, run download cell first')
if not BASE_CHECKPOINT.is_file():
    raise FileNotFoundError(f'base checkpoint not found: {BASE_CHECKPOINT}')
dataset_sha256 = require_sha256(DATASET, DATASET_SHA256, 'dataset')
base_checkpoint_sha256 = require_sha256(
    BASE_CHECKPOINT, BASE_CHECKPOINT_SHA256, 'base checkpoint'
)
base_epoch = checkpoint_epoch(BASE_CHECKPOINT)

if smoke:
    additional_epochs = min(additional_epochs, 2)
    save_every = 1
    epoch_size = min(epoch_size, 500_000)
    num_workers = 2
    print(">>> SMOKE TEST:", additional_epochs, "superbatches of", epoch_size, "positions")

output_name = f'{RUN_NAME}-smoke' if smoke else RUN_NAME
OUT_ROOT = (DRIVE_DIR / output_name).resolve()
try:
    BASE_CHECKPOINT.relative_to(OUT_ROOT)
except ValueError:
    pass
else:
    raise ValueError('BASE_CHECKPOINT must be outside this run output directory')
target_max_epochs = derive_target_max_epochs(base_epoch, additional_epochs)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = OUT_ROOT / 'fine-tune-manifest.json'
manifest = {
    'schema': 1,
    'run_name': output_name,
    'smoke': smoke,
    'nnue_repo': str(NNUE_REPO),
    'nnue_revision': nnue_revision,
    'dataset': str(DATASET),
    'dataset_sha256': dataset_sha256,
    'base_checkpoint': str(BASE_CHECKPOINT),
    'base_checkpoint_sha256': base_checkpoint_sha256,
    'base_epoch': base_epoch,
    'additional_epochs': additional_epochs,
    'target_max_epochs': target_max_epochs,
    'features': 'Full_Threats+HalfKAv2_hm^',
    'l1': 1024,
    'l2': 32,
    'l3': 32,
    'batch_size': batch_size,
    'epoch_size': epoch_size,
    'save_every': save_every,
    'num_workers': num_workers,
    'accelerator': 'cuda',
    'precision': os.environ['LIGHTNING_PRECISION'],
}
manifest_preexisting = manifest_path.exists()
if manifest_preexisting:
    with manifest_path.open(encoding='utf-8') as stream:
        existing_manifest = json.load(stream)
    if existing_manifest != manifest:
        raise ValueError(f'run manifest does not match requested fine-tune: {manifest_path}')
elif any(OUT_ROOT.iterdir()):
    raise ValueError(f'non-empty output directory has no fine-tune manifest: {OUT_ROOT}')
else:
    atomic_write_json(manifest_path, manifest)

resume = BASE_CHECKPOINT
if RESUME_CHECKPOINT:
    resume = Path(RESUME_CHECKPOINT).expanduser().resolve()
    if not resume.is_file():
        raise FileNotFoundError(f'resume checkpoint not found: {resume}')
    try:
        resume.relative_to(OUT_ROOT)
    except ValueError as error:
        raise ValueError('RESUME_CHECKPOINT must be inside this run output directory') from error
elif manifest_preexisting and any(OUT_ROOT.rglob('*.ckpt')):
    raise ValueError('existing run checkpoints require an explicit RESUME_CHECKPOINT')
resume_epoch = base_epoch if resume == BASE_CHECKPOINT else checkpoint_epoch(resume)
validate_resume_epoch(base_epoch, resume_epoch, target_max_epochs)
resume_sha256 = (
    base_checkpoint_sha256 if resume == BASE_CHECKPOINT else sha256_file(resume)
)

cmd = [
    sys.executable, "train.py", str(DATASET),
    "--features", "Full_Threats+HalfKAv2_hm^",
    "--l1", "1024", "--l2", "32", "--l3", "32",
    "--batch-size", str(batch_size),
    "--epoch-size", str(epoch_size),
    "--max-epochs", str(target_max_epochs),
    "--network-save-period", str(save_every),
    "--save-top-k", "-1",
    "--num-workers", str(num_workers),
    "--default-root-dir", str(OUT_ROOT),
    "--accelerator", "cuda",
]
cmd += ["--resume-from-checkpoint", str(resume)]
print(">>> base epoch", base_epoch, "target max epochs", target_max_epochs)
print(">>> resume from", resume, "sha256", resume_sha256)
print("$", " ".join(cmd), "\n")

launch_path = OUT_ROOT / 'active-launch.json'
if launch_path.exists():
    with launch_path.open(encoding='utf-8') as stream:
        previous_launch = json.load(stream)
    if previous_launch.get('status') in ('completed', 'failed', 'interrupted'):
        archive_launch(OUT_ROOT, previous_launch)
        atomic_write_json(OUT_ROOT / 'last-launch.json', previous_launch)
        launch_path.unlink()
    else:
        required_identity = ('pid', 'hostname', 'process_start_ticks', 'command', 'output_root')
        if previous_launch.get('status') not in ('starting', 'running'):
            raise RuntimeError(f'active launch has invalid status: {previous_launch}')
        if any(previous_launch.get(field) is None for field in required_identity):
            raise RuntimeError(f'active launch identity is incomplete: {launch_path}')
        if previous_launch['hostname'] != socket.gethostname():
            raise RuntimeError(f'active launch belongs to another host: {launch_path}')
        if previous_launch['output_root'] != str(OUT_ROOT):
            raise RuntimeError(f'active launch belongs to another output directory: {launch_path}')
        active_identity = process_identity(previous_launch['pid'])
        if active_identity is not None:
            same_start = active_identity['start_ticks'] == previous_launch['process_start_ticks']
            same_command = active_identity['command'] == previous_launch['command']
            if same_start and same_command:
                raise RuntimeError(f'trainer is still running with pid {previous_launch["pid"]}')
        if not recover_dead_launch:
            raise RuntimeError('trainer is gone; inspect it and set RECOVER_DEAD_LAUNCH=True')
        previous_launch['status'] = 'abandoned'
        previous_launch['recovered_at'] = dt.datetime.now(dt.timezone.utc).isoformat()
        archive_launch(OUT_ROOT, previous_launch)
        launch_path.unlink()
launch = {
    'schema': 1,
    'launch_id': uuid.uuid4().hex,
    'created_at': dt.datetime.now(dt.timezone.utc).isoformat(),
    'status': 'starting',
    'hostname': socket.gethostname(),
    'output_root': str(OUT_ROOT),
    'resume_checkpoint': str(resume),
    'resume_checkpoint_sha256': resume_sha256,
    'resume_epoch': resume_epoch,
    'command': cmd,
}
atomic_write_json(launch_path, launch)

process = subprocess.Popen(
    cmd,
    cwd=NNUE_REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
    env=os.environ.copy(),
    start_new_session=True,
)
identity = process_identity(process.pid)
if identity is None:
    return_code = process.wait()
    launch['pid'] = process.pid
    launch['status'] = 'failed' if return_code else 'completed'
    launch['return_code'] = return_code
    launch['completed_at'] = dt.datetime.now(dt.timezone.utc).isoformat()
    atomic_write_json(launch_path, launch)
    archive_launch(OUT_ROOT, launch)
    atomic_write_json(OUT_ROOT / 'last-launch.json', launch)
    launch_path.unlink()
    raise RuntimeError('trainer exited before its process identity was recorded')
launch['status'] = 'running'
launch['pid'] = process.pid
launch['process_start_ticks'] = identity['start_ticks']
launch['command'] = identity['command']
atomic_write_json(launch_path, launch)

def terminate_process_group(process):
    if process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()

def finish_launch(status, return_code):
    launch['status'] = status
    launch['return_code'] = return_code
    launch['completed_at'] = dt.datetime.now(dt.timezone.utc).isoformat()
    atomic_write_json(launch_path, launch)
    archive_launch(OUT_ROOT, launch)
    atomic_write_json(OUT_ROOT / 'last-launch.json', launch)
    with launch_path.open(encoding='utf-8') as stream:
        active_launch = json.load(stream)
    if active_launch.get('launch_id') != launch['launch_id']:
        raise RuntimeError('active launch changed while trainer was running')
    launch_path.unlink()

try:
    for line in process.stdout:
        print(line, end='')
except KeyboardInterrupt:
    terminate_process_group(process)
    finish_launch('interrupted', process.returncode)
    raise

return_code = process.wait()
finish_launch('failed' if return_code else 'completed', return_code)
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)